In [1]:
import spacy
from spacy.pipeline import Sentencizer
from spacy.language import Language
import os
import json
import pandas as pd
import re
import requests
from periodictable import elements
from pathlib import Path
from pypdf import PdfReader as pr
import pubchempy as pcp
import time
from biocyc import biocyc
import urllib.parse
from xml.etree import ElementTree as ET
import random
import bidict

import warnings
warnings.filterwarnings("ignore")

In [2]:
#os.chdir('..\\dissertation DLC content')
#linker = Ontology('OntoBiotope_BioNLP-OST-2019.obo')

path_to_scispacy_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\en_ner_bionlp13cg_md-0.5.4\\en_ner_bionlp13cg_md\\en_ner_bionlp13cg_md-0.5.4')
path_to_custom_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_ner')

scispacy_model = spacy.load(path_to_custom_model, disable=['parser'])

@Language.component('custom_sentencizer')
def create_custom_sentencizer(doc):
    for ent in doc.ents:
        for t in range(len(ent)-1):
            doc[ent.start+t+1].is_sent_start = False
    return doc

sentenciser = scispacy_model.add_pipe('custom_sentencizer', name='custom_sentencizer', after='sentencizer')

#microbial_chemical_model = spacy.load(path_to_scispacy_model)

# <font size=4>hidden</font>

In [3]:
%%script False

print(scispacy_model.pipe_names)

#linker = pyobo.from_obo_path('..\\dissertation DLC content\\chebi_lite.obo', version='1.2')
kb = KnowledgeBase('..\\dissertation DLC content\\chebi_lite.json')

linker = scispacy_model.add_pipe('entity_linker', after='ner')
linker.set_kb(kb)

onto = pyobo.fr('chebi_core')

#update this here to try and download a more relevant knowledge base
""" !python -m spacy_entity_linker "download_knowledge_base"

from spacy.kb import InMemoryLookupKB

def create_kb(vocab):
    kb = InMemoryLookupKB(vocab, entity_vector_length=128)
    return kb

linker = scispacy_model.add_pipe('entityLinker', after='ner') """

Couldn't find program: 'False'


# <font size=4>not hidden</font>

In [4]:
with open('list_files\\utilizations_list.json', 'r', encoding='utf-8') as f:
    utilizations = list(json.load(f))

basic_utilizations = ['fermentation', 'hydrolyze', 'hydrolyse', 'reduce', 'reduces', 'reduced', 'produce', 'production', 'produces', 'consume', 'consumes', 'consumption', 'consuming', 'producing', 'grow', 'growth', 'oxidise', 'oxidize', 'consumed', 'produced', 'convert', 'converts', 'converting', 'converted', 'conversion']

print('|'.join(basic_utilizations))

util_mappings = {
    'production': 'PRODUCES',
    'produce': 'PRODUCES',
    'produces': 'PRODUCES'
}
    
custom_labels = ['MICROBE_NAME', 'MICROBE', 'SIMPLE_CHEMICAL', 'CHEMICAL']

with open('list_files\\genus_names.json', 'r', encoding='utf-8') as f:
    genera_unfiltered = list(json.load(f))
    
with open('list_files\\food_list.json', 'r', encoding='utf-8') as f:
    food_list = list(json.load(f))
    
genus_names = []

for gen in genera_unfiltered:
    if 'Candidatus' in gen:
        gen_split = gen.split(' ')
        genus_names.append(gen_split[1])
    else:
        genus_names.append(gen)
        
with open('list_files\\microbes_genus_species.json', 'r', encoding='utf-8') as f:
    microbes_genus_species = list(json.load(f))
    
not_chemicals = ['flavour', 'flavor', 'flavours', 'flavors', 'carbohydrate', 'carbohydrates', 'vitamin', 'vitamins', 'aroma', 'aromas', 'flavonoid', 'flavonoids', 'co2', 'o2', 'liquid', 'oxygen', 'nitrogen', 'dioxygen', 'carbon dioxide'] + genus_names + food_list
print(len(not_chemicals))
    
g_species = []
genus_species = []
for gs in microbes_genus_species:
    genus, species = gs.split(' ')
    g_species.append(str(genus[0] + '. ' + species))
    genus_species.append((genus, species))
    
print(len(g_species), len(set(g_species)))
    
with open('list_files\\species_names.json', 'r', encoding='utf-8') as f:
    species_names = list(set(json.load(f)))
    
with open('list_files\\basionym_realname_mapping.json', 'r', encoding='utf-8') as f:
    basionym_realname_mapping = dict(json.load(f))

with open('list_files\\realname_basionym_mapping.json', 'r', encoding='utf-8') as f:
    realname_basionym_mapping = dict(json.load(f))

#basionym_realname_mapping_bidict = bidict.bidict(basionym_realname_mapping)

""" with open('list_files\\pcp_metabolite_production_mapping.json', 'r', encoding='utf-8') as f:
    pcp_metabolite_production_mapping = dict(json.load(f))

with open('list_files\\pcp_metabolite_utilisation_mapping.json', 'r', encoding='utf-8') as f:
    pcp_metabolite_utilisation_mapping = dict(json.load(f))

pcp_mappings = {
    'production': pcp_metabolite_production_mapping,
    'utilisation': pcp_metabolite_utilisation_mapping
}

with open('list_files\\metabolite_production.json', 'r', encoding='utf-8') as f:
    metabolite_production = dict(json.load(f))

with open('list_files\\metabolite_utilisation.json', 'r', encoding='utf-8') as f:
    metabolite_utilisation = dict(json.load(f)) """

with open('list_files\\noresolve_chemicals_mapping.json', 'r', encoding='utf-8') as f:
    noresolve_chemicals_mapping = dict(json.load(f))

with open('list_files\\biocyc_db_mappings.json', 'r', encoding='utf-8') as f:
    biocyc_db_mappings = dict(json.load(f))

with open('list_files\\gbif_resolved_mappings.json', 'r', encoding='utf-8') as f:
    gbif_resolved_mappings = dict(json.load(f))

fermentation|hydrolyze|hydrolyse|reduce|reduces|reduced|produce|production|produces|consume|consumes|consumption|consuming|producing|grow|growth|oxidise|oxidize|consumed|produced|convert|converts|converting|converted|conversion
14938
24692 21518


Fixed mappings

In [5]:
microbe_shortforms_known_mappings = {
    'S. cerevisiae': 'Saccharomyces cerevisiae',
    'A. niger': 'Aspergillus niger',
    'S. fragilis': 'Saccharomyces fragilis',
    'L. hilgardii': 'Lentilactobacillus hilgardii'
}

numbers_map = {
    0: 'zero',
    1: 'one',
    2: 'two',
    3: 'three',
    4: 'four',
    5: 'five',
    6: 'six',
    7: 'seven',
    8: 'eight',
    9: 'nine'
}

fixed_chemical_resolutions = {
    'l-lactic acid': 'lactic acid',
    'l-lactate': 'lactic acid',
    'lactate': 'lactic acid',
    'l-malic acid': 'malic acid',
    'l-malate': 'malic acid',
    '(s)-lactate': 'lactic acid'
}

def preprocess_label(label):
    if label[0].isnumeric():
        label = label.replace(label[0], numbers_map[int(label[0])])
    replace_chars = ["-", "(", ")", " ", ".", ",", "\'", "/", "\"", ";", "&", ":", ";", "|", "%", "’", "?", "!", "«", "+", "（", "）", "？", "，"]
    for char in replace_chars:
        label = label.replace(char, '')
    return label

def preprocess_name(name):
    new_name = name
    name.replace('(', '').replace(')', '')
    replace_chars = [' ', '[', ']']
    new_name = new_name.strip()
    return new_name

def process_html(text):
    html_tags = re.findall('</?[A-Za-z]+>', text)

    for tag in html_tags:
        text = text.replace(tag, '')

    return text

biocyc_chemical_resolves = {
    'lactic acid': 'r-lactic acid',
    'glucose': 'd-glucose',
    'malic acid': 'l-malic acid',
    'fructose': 'd-fructose',
    'lactate': 'd-lactate',
    'malate': 'l-malic acid'
}

# <h5>Create cypher commands</h5>

In [6]:
#%%script False

import json
import re
from neo4j import GraphDatabase

numbers_map = {0: 'zero', 1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine'}

def preprocess_label(label):
    if label[0].isnumeric():
        label = label.replace(label[0], numbers_map[int(label[0])])
    replace_chars = ["-", "(", ")", " ", ".", ",", "\'", "/", "\"", ";", "&", ":", ";", "|", "%", "’", "?", "!", "«", "+", "（", "）", "？", "，", "<", ">"]
    for char in replace_chars:
        label = label.replace(char, '')
    return label

def preprocess_name(name):
    replace_chars = ['[', ']', "'", '(', ')', '\"', '{', '}']
    for char in replace_chars:
        name = name.replace(char, '')
    return name

def process_html(text):
    html_tags = re.findall('</?[A-Za-z]+>', text)

    for tag in html_tags:
        text = text.replace(tag, '')

    return text

with open('list_files\\saved_relations.json', 'r', encoding='utf-8') as f:
    saved_relations = json.load(f)

with open("list_files\\entity_label_map.json", 'r') as f:
    entity_label_map = dict(json.load(f))


#creates relations and adds them to neo4j
def add_cypher_relations_neo(relations):
    with open('..\\dissertation DLC content\\go away\\passwords.json', 'r') as f:
        passwords = dict(json.load(f))

    neo4j_pw = passwords['neo4j']

    del passwords

    # URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
    URI = "neo4j://127.0.0.1:7687"
    AUTH = ("neo4j", neo4j_pw)

    driver = GraphDatabase.driver(URI, auth=AUTH, max_connection_lifetime=3600)

    processed_relations = set()

    with driver.session() as session:

        for ent1type, ent1, rel, ent2type, ent2, value in relations:

            if not value:
                continue
            ent1name = process_html(preprocess_name(ent1))
            ent1label = preprocess_label(ent1name)

            ent2name = process_html(preprocess_name(ent2))
            ent2label = preprocess_label(ent2name)

            if not entity_label_map.get(ent1label):
                #ent1string = f"MERGE ({ent1label}:{ent1type}" + r"{" + f"name: '{ent1name}" + r"'})"
                ent1string = f"MERGE ({ent1label}:{ent1type}" + r"{" + f"customId: '{ent1label}'" f", name: '{ent1name}" + r"'})"
                if ent1string in processed_relations:
                    continue
                processed_relations.add(ent1string)

                entity_label_map[ent1label] = ent1name

                with open('all_entity_commands.cypher', 'a', encoding='utf-8') as f:
                    f.write(ent1string)
                    f.write('\n')

                session.run(ent1string)

            if not entity_label_map.get(ent2label):
                #ent2string = f"MERGE ({ent2label}:{ent2type}" + r"{" + f"name: '{ent2name}" + r"'})"
                ent2string = f"MERGE ({ent2label}:{ent2type}" + r"{" + f"customId: '{ent2label}'" + f", name: '{ent2name}" + r"'})"
                if ent2string in processed_relations:
                    continue
                processed_relations.add(ent2string)

                entity_label_map[ent2label] = ent2name

                with open('all_entity_commands.cypher', 'a', encoding='utf-8') as f:
                    f.write(ent2string)
                    f.write('\n')

                session.run(ent2string)

            relstring = f"MERGE ({ent1label})-[:{rel}]->({ent2label})"

            if relstring in processed_relations:
                continue

            processed_relations.add(relstring)

            with open('all_relation_commands.cypher', 'a', encoding='utf-8') as f:
                f.write(relstring)
                f.write('\n')

            relstringneo =  f"""MATCH (a) WHERE a.customId = '{ent1label}'
                                MATCH (b) WHERE b.customId = '{ent2label}'
                                MERGE (a)-[:{rel}]->(b)
                             """
            
            session.run(relstringneo)

In [15]:
with open('list_files\\paper_number.json', 'r') as f:
    paper_no = int(json.load(f))

with open("list_files\\entity_label_map.json", 'r') as f:
    entity_label_map = dict(json.load(f))

with open('list_files\\reaction_enzyme_mapping.json', 'r', encoding='utf-8') as f:
    reaction_enzyme_mapping = dict(json.load(f))

with open('list_files\\biocyc_common_name_mapping.json', 'r', encoding='utf-8') as f:
    biocyc_common_name_mapping = dict(json.load(f))

#tuple of (microbe, consume/produce, chemical) or (chemical, become, chemical), and whether its true or false
with open('list_files\\saved_relations.json', 'r', encoding='utf-8') as f:
    saved_relations = list(json.load(f))

email = 'puskas@andrews.edu'

with open('..\\dissertation DLC content\\go away\\passwords.json', 'r') as f:
    passwords = json.load(f)
    password = passwords.get('metacyc_pw')

with open('list_files\\pcp_chemical_verification.json', 'r', encoding='utf-8') as f:
    chemical_verification = dict(json.load(f))

with open('list_files\\microbe_chemical_db_fails.json', 'r') as f:
    microbe_chemical_db_fails = dict(json.load(f))

files_mapping = {
    'saved_relations': saved_relations,
    'reaction_enzyme_mapping': reaction_enzyme_mapping,
    'pcp_chemical_mapping': chemical_verification,
    'biocyc_common_name_mapping': biocyc_common_name_mapping,
    'gbif_resolved_mappings': gbif_resolved_mappings,
    'microbe_chemical_db_fails': microbe_chemical_db_fails,
    'entity_label_map': entity_label_map
}

new_relations = []

def add_relation(relation):
    saved_relations.append(relation)
    new_relations.append(relation)

daily_requests = True

session = requests.Session()

session.post('https://websvc.biocyc.org/credentials/login/', data={'email':email, 'password':password}, headers={'User-Agent': 'A University of Bath student. I am doing a dissertation project in Natural Language Processing, where I identify named entities such as microbes and chemicals in academic papers, and the relations between them. I am using the database for the relation extraction element, and whether a microbe consumes or produces a particular chemical.'})

for ind, file in enumerate(os.listdir("..\\dissertation DLC content\\fermentation_papers_preprocessed")[paper_no:paper_no+2]):
    if not daily_requests:
        break
    microbe_shortform_mappings = {}
    print(ind)
    #keep track of longterm names of all microbes seen so far
    filename = os.fsdecode(file)
    if filename.endswith('json'):
        with open(f"..\\dissertation DLC content\\fermentation_papers_preprocessed\\{filename}", 'r', encoding='utf-16') as f:
            text = f.read()
    elif filename.endswith('pdf'):
        try:
            print('pdf!')
            path = Path(os.getcwd() + "\\fermentation_papers\\" + filename)
            reader = pr(path)
            text = ""
            for page in reader.pages:
                text = text + page.extract_text()
        except:
            continue
    else:
        continue
    
    def resolve_basionym(name):
        resolved_name = name
        basionym_mapping_name = basionym_realname_mapping.get(name)
        if basionym_mapping_name:
            resolved_name = basionym_mapping_name
        else:
            if gbif_resolved_mappings.get(name):
                resolved_name = gbif_resolved_mappings[name]
            else:
                results = requests.get(f"https://api.gbif.org/v1/species/search?q={str(resolved_name.replace(' ', '+'))}").json().get('results')
                time.sleep(0.1)
                if results:
                    gbif_name = results[0].get('species')
                    if gbif_name:
                        gbif_resolved_mappings[name] = gbif_name
                        resolved_name = gbif_name
                else:
                    gbif_resolved_mappings[name] = None
        return resolved_name
    
    def pcp_chemical_verification(chem):
        if chem in chemical_verification.keys():
            if chemical_verification[chem] == None:
                return None
        try:
            comps = pcp.get_compounds(chem, 'name')
            time.sleep(0.2)
        except:
            #assume that its not a chemical
            chemical_verification[chem] = None
            return None
        if not comps == None:
            try:
                synonyms = comps[0].synonyms
                return chem
            except:
                chemical_verification[chem] = None
                return None

        chemical_verification[chem] = None #if nothing has happened by this point just put 'None'
        return None
    
    paper_processed = scispacy_model(text)

    real_chem_found = False

    all_mic_ents = []

    #instead get only ONE verified chemical, rather than verifying all of them to prove that chemicals exist in the paper. would save a buncha time
    #also since we are already iterating through all ents why not grab the microbes too
    for ent in paper_processed.ents:
        if 'MICROBE' in ent.label_:
            all_mic_ents.append(('MICROBE_NAME', ent.text, ent.start_char, ent.end_char))
        if not real_chem_found:
            if 'CHEMICAL' in ent.label_:
                if pcp_chemical_verification(ent.text) != None:
                    real_chem_found = True

    #no chemicals with which to form a relation
    if not real_chem_found:
        print("no real chemicals found")
        continue

    if len(all_mic_ents) == 0:
        print('no microbes found')
        continue

    print("first 5 microbes:", all_mic_ents[:5])

    #do a first pass of the paper to identify microbe shortforms
    all_mic_shapes_raw = [mic.shape_ for mic in paper_processed]
    mic_shapes = [(name, ' '.join([msr[:4] for msr in all_mic_shapes_raw[s:e]]), sc, ec) for s, e, name, sc, ec in [(m.start, m.end, m.text, m.start_char, m.end_char) for m in paper_processed.ents if 'MICROBE' in m.label_]]
    
    for name, shape, s, e in mic_shapes:
        if shape == 'Xxxx xxxx':
            genus, species = name.split(' ')
            shortform_name = genus[0] + '. ' + species
            if not microbe_shortform_mappings.get(shortform_name):
                #normalize microbe name before adding
                resolved_basionym = resolve_basionym(name)
                if resolve_basionym:
                    microbe_shortform_mappings[shortform_name] = resolved_basionym

        elif shape == 'X. xxxx' or shape == 'Xx. xxxx' or shape == 'Xxx. xxxx':
            if not microbe_shortform_mappings.get(name):
                #add that entry in to indicate we have seen the shortform before the longform, in case the shortform doesnt get seen again, sometimes the microbe names are weird
                microbe_shortform_mappings[name] = None

    #keep track of previous sentence and entities for which to merge with, if there are not enough to form relations with
    merge_sentences = []
    last_sent_length = 0
    merge_ents_resolved = []
    merge_update = False

    print("finished processing paper")

    for sent_raw in paper_processed.sents:
        if not daily_requests:
            break

        mic_ents_unfiltered = [(mic.label_, mic.text, mic.start_char, mic.end_char) for mic in sent_raw.ents if 'MICROBE' in mic.label_]

        #these are filtered but unresolved, so we keep the original, un-normalized names of the microbes, but they are filtered. this is done to remove people's names which
        #are misidentified as microbes
        mic_ents_filtered = []
        mic_shapes_raw = [mic.shape_ for mic in sent_raw]

        mic_ents_full_detail = [(m.start-sent_raw.start, m.end-sent_raw.end, m.text, m.start_char-sent_raw.start_char, m.end_char-sent_raw.start_char) for m in sent_raw.ents if 'MICROBE' in m.label_]

        mic_shapes = [(name, ' '.join([msr[:4] for msr in mic_shapes_raw[s:e]]), sc, ec) for s, e, name, sc, ec in mic_ents_full_detail]
        mic_ents_resolved = []

        for name, shape, s, e in mic_shapes:
            if shape == 'X. xxxx' or shape == 'Xx. xxxx' or shape == 'Xxx. xxxx':
                alternate_name = None
                if shape == 'Xxx. xxxx':
                    print("weird name:", name)
                #check the map to see if theres anything in there
                shortform_match_name = microbe_shortform_mappings.get(name)
                #add the resolved name to the array
                if shortform_match_name:
                    resolved_name = resolve_basionym(shortform_match_name[0] + " " + shortform_match_name[1])
                    alternate_name = resolved_name
                    mic_ents_resolved.append(('MICROBE_NAME', resolved_name, s, e))
                else:
                    #try with the known mappings (obviously not gonna do them all, just the most common ones)
                    known_name = microbe_shortforms_known_mappings.get(name)
                    if known_name:
                        alternate_name = known_name
                        microbe_shortform_mappings[name] = known_name
                        mic_ents_resolved.append(('MICROBE_NAME', known_name, s, e))
                    else:
                        #perhaps match against the list of genera and species to see which is the most probable? like start with the first letter and figure shit out from there
                        #like check the species name and check it against the full names, and see if theres any genera which match up
                        #but what if there are multiple candidates?
                        
                        full_genus, full_species = name.split(' ') #genus is first letter, species is 'xxxx' bit after
                        g_species = full_genus[0] + '. ' + full_species
                        #originally dont match with genus to check for potential name changes
                        matched_species = [(gen, spec) for gen, spec in genus_species if spec == full_species]
                        
                        matched_name = None
                        
                        if len(matched_species) == 1:
                            matched_name = matched_species[0]
                        elif len(matched_species) == 2:
                            gen1, spec1 = matched_species[0]
                            resolved_basionym_1 = resolve_basionym(str(gen1 + '+' + spec1))
                            
                            gen2, spec2 = matched_species[1]
                            resolved_basionym_2 = resolve_basionym(str(gen2 + '+' + spec2))
                            
                            if resolved_basionym_1 == resolved_basionym_2:
                                matched_name = (gen1, spec1) #found match!
                                microbe_shortform_mappings[name] = gen1 + '+' + spec1
                            else:
                                #TODO: figure something out idk
                                pass
                        else:
                            #match with genus name rather than just species
                            matched_genus = [(gen, spec) for gen, spec in matched_species if gen[0] == full_genus[0]]
                            if len(set(matched_genus)) == 1:
                                matched_name = matched_genus[0] #found match!
                            else:
                                #figure something out idk, maybe the same thing as above
                                pass
                                
                        alternate_name = matched_name
                               
                        #resolve name and add mapping 
                        if matched_name:
                            resolved_basionym = resolve_basionym(' '.join(matched_name))
                            mic_ents_resolved.append(('MICROBE_NAME', str(resolved_basionym), s, e))
                            #only add the original name to the shortform mapping, not the resolved names
                            microbe_shortform_mappings[name] = matched_name
                        
                #sometimes an entity is flagged as a microbe when it is actually not. however, i want to keep the original microbe names for the relation extraction part-
                #the llm will see the context sentence and if we give it the resolved entities it will not know how to match them.
                #so only add to the filtered list if an alternate, resolved name exists.
                if alternate_name:
                    mic_ents_filtered.append(('MICROBE_NAME', name, s, e))
                        
            elif shape == 'Xxxx xxxx':
                resolved_basionym = resolve_basionym(name)
                name_split = name.split(' ')
                shortform_name = name_split[0][0] + '. ' + name_split[1]
                #then add it to 'resolved'
                mic_ents_resolved.append(('MICROBE_NAME', resolved_basionym, s, e))
                mic_ents_filtered.append(('MICROBE_NAME', name, s, e))

        #chem_ents = [(sci.label_, sci.text.lower(), sci.start_char, sci.end_char) for sci in sent_raw.ents if 'CHEMICAL' in sci.label_ and pcp_chemical_verification(sci.text.lower()) != None]
        chem_ents = []

        for sci in sent_raw.ents:
            sci_text = sci.text.lower()
            if 'CHEMICAL' in sci.label_ and sci_text not in not_chemicals + [text.lower() for l, text, s, e in mic_ents_filtered]:
                #if pubchempy identifies it as a chemical, put it forward, will reduce having to retrieve a chemical id with biocyc which will use up a daily query
                #because the free account only gets a limited no. biocyc queries per day
                if pcp_chemical_verification(sci_text):
                    if biocyc_chemical_resolves.get(sci_text):
                        chem_ents.append(('CHEMICAL', biocyc_chemical_resolves[sci_text], sci.start_char, sci.end_char))
                    else:
                        chem_ents.append(('CHEMICAL', sci_text, sci.start_char, sci.end_char))
                else:
                    continue

        ents_and_labels = [(label, text) for label, text, start, end in mic_ents_resolved + chem_ents if label in custom_labels and len(text)>=4]
        #ent_labels = [label for label, t in ents_and_labels]
        #ents_and_labels_resolved = sorted(ents_and_labels_resolved, key= lambda tup: tup[2])
            
        if not merge_sentences:
            #if there are entities which are not empty
            if ents_and_labels:
                merge_sentences.append(sent_raw.text)
                merge_ents_resolved.append(ents_and_labels)
                merge_update = True

        #but if there is a sentence awaiting merging
        else:
            #if there are any entities to merge with, check that they arent all the same type
            if ents_and_labels:
                #premerged_labels = [label for ent_list in merge_ents_resolved for label, text in ent_list] + [l for l, t in ents_and_labels]

                merge_ents_resolved.append(ents_and_labels)
                merge_sentences.append(sent_raw.text)
                merge_update = True

                merge_labels = set([label for label, t in ents_and_labels])

        while len(merge_sentences) > 3:
            merge_sentences.pop(0)
            merge_ents_resolved.pop(0)
            merge_update = True

        #all_merged_labels_resolved = [label for label,t,s,e in merge_ents_resolved]

        all_merge_ents = [ent for ent_list in merge_ents_resolved for ent in ent_list]
        all_merge_labels = [label for label, e in all_merge_ents]
      
        #checking to see if the merged ents can be written to the file, or if we should just keep merging until theres enough to form a 'relation'
        if len(all_merge_ents) >= 3 and len(set(all_merge_labels)) > 1 and merge_update:
            microbes = list(set([mic for label,mic in all_merge_ents if 'MICROBE' in label]))
            no_microbes = len(microbes)
            chemicals = list(set([text for label, text in all_merge_ents if 'CHEMICAL' in label])) #the types of chemicals identified
            no_chemicals = len(chemicals)

            condition1 = no_microbes >= 1 and no_chemicals >= 1 #MICROBE CONSUMES/PRODUCES CHEMICAL
            if condition1:

                def process_biocyc_request(request):
                    biocyc_resp = session.get(request)
                    if biocyc_resp.status_code == 200:
                        return biocyc_resp
                    else:
                        daily_requests = False
                        return 'error'

                merge_update = False
                    
                def process_comp(comp):
                    replace_chars = [' ', '.', '?', '!', ',', '\\', '/']
                    for char in replace_chars:
                        comp = comp.replace(char, '')
                    return comp
                
                print(microbes, chemicals)

                def get_common_name(search_chem):
                    search_chem = process_comp(chem)

                    if biocyc_common_name_mapping.get(database, {}).get(search_chem):
                        return biocyc_common_name_mapping[database][search_chem]
                    
                    #if nothing comes up in compounds search assume its an enzyme which was misidentified as a chemical
                    chem_resp = process_biocyc_request(f'https://websvc.biocyc.org/{database}/name-search?object={search_chem}&class=Compounds&fmt=json')
                    if chem_resp == 'error':
                        return None, None

                    time.sleep(1)

                    if not biocyc_common_name_mapping.get(database):
                        biocyc_common_name_mapping[database] = {}

                    try:
                        chem_resp = chem_resp.json()
                    except:
                        print("json error:")
                        biocyc_common_name_mapping[database][search_chem] = (None, None)
                        return None, None

                    if not chem_resp.get('RESULTS'):
                        #nothing to see here!
                        biocyc_common_name_mapping[database][chem] = (None, None)
                        return None, None

                    chem_resp = chem_resp.get('RESULTS')

                    chem_ids = [result.get('OBJECT-ID') for result in chem_resp]
                    chem_common_name = chem_resp[0].get('COMMON-NAME')
                    chem_common_name = process_html(chem_common_name)

                    biocyc_common_name_mapping[database][chem] = (chem_ids, chem_common_name)
                    biocyc_common_name_mapping[database][search_chem] = (chem_ids, chem_common_name)

                    return chem_ids, chem_common_name

                for microbe in microbes:
                    dbs = biocyc_db_mappings.get(microbe)

                    if not dbs:
                        if realname_basionym_mapping.get(microbe):
                            try:
                                dbs = biocyc_db_mappings[realname_basionym_mapping.get(microbe)]
                            except:
                                continue
                        else:
                            continue

                    print("microbe:", microbe)

                    if not daily_requests:
                        break

                    for chem in chemicals:
                        #bacdive stuff if biocyc doesnt work out

                        """ #testing for utilisations
                        def test_metabolites(method, relation):
                            synonyms = pcp_mappings[method].get(chem)
                            print(synonyms)
                            if not synonyms:
                                return
                            metabolite_found = False
                            for syn in synonyms:
                                if syn in metabolite_utilisation.get(microbe):
                                    metabolite_found = True
                                    break
                            if metabolite_found:
                                print("relation found!")
                                new_relations[('MICROBE', microbe, relation, 'CHEMICAL', chem)] = True

                        test_metabolites('utilisation', 'CONSUMES')
                        test_metabolites('production', 'PRODUCES') """

                        if not daily_requests:
                            break

                        for filename, variable in files_mapping.items():
                            with open(f'list_files\\{filename}.json', 'w', encoding='utf-8') as f:
                                json.dump(variable, f)

                        #a microbe may have multiple databases for strains, but if the strain is not included just pick a random one and hope for the best lmao
                        #there would have to be a reason they dont include the strain in the paper, right?
                        if len(dbs) == 1:
                            database = list(dbs.values())[0]
                        else:
                            database = random.choice(list(dbs.values()))

                        if biocyc_common_name_mapping.get(chem):
                            chem_ids, chem_common_name = biocyc_common_name_mapping[database][chem]
                        else:
                            chem_ids, chem_common_name = get_common_name(chem)

                        if not chem_ids:
                            #try bacdive?
                            continue

                        if ('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', chem_common_name, True) in saved_relations or ('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', chem_common_name, False) in saved_relations:
                            print("relation already found. continue")
                            continue

                        chem_id = None #these are the ids for which reactions exist
                        good_reactions_xml = None

                        for id in chem_ids:
                            if not microbe_chemical_db_fails.get(microbe):
                                microbe_chemical_db_fails[microbe] = {}

                            if not microbe_chemical_db_fails[microbe].get(database):
                                microbe_chemical_db_fails[microbe][database] = []

                            if id in microbe_chemical_db_fails[microbe].get(database, []):
                                #maybe try the other database
                                new_dbs = [(db, ids) for db, ids in biocyc_db_mappings[microbe].items() if db != database and id not in microbe_chemical_db_fails[microbe].get(database, [])]
                                if new_dbs:
                                    database = new_dbs[0][0]
                                else:
                                    continue

                            try:
                                print("testing database")
                                chem_reactions_resp = process_biocyc_request(f"https://websvc.biocyc.org/apixml?fn=reactions-of-compound&id={database}:{id}&detail=full")
                                if chem_reactions_resp == 'error':
                                    print("error!")
                                    break
                                time.sleep(0.8)
                                reactions_xml = ET.fromstring(chem_reactions_resp.text)
                                try:
                                    num_results = str(reactions_xml.find('metadata').find('num_results').text)
                                except:
                                    microbe_chemical_db_fails[microbe][database].append(id)
                                    continue

                                if num_results == "0" or num_results == None:
                                    microbe_chemical_db_fails[microbe][database].append(id)
                                    continue

                                else:
                                    #check to see if there exists a chemical in 'left' or 'right with the same id
                                    all_comp_frameids = [comp.get('frameid') for comp in reactions_xml.iter() if comp.tag == 'Compound']
                                    if id in all_comp_frameids:
                                        chem_id = id
                                        good_reactions_xml = reactions_xml
                                        break
                                    else:
                                        print("all these compounds and not one in which the chemical id appears in... wtf is this")
                                        microbe_chemical_db_fails[microbe][database].append(id)
                                        continue
                            except:
                                microbe_chemical_db_fails[microbe][database].append(id)
                                continue

                        if not chem_id or good_reactions_xml == None:
                            saved_relations.append(('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', chem_common_name, False))
                            saved_relations.append(('MICROBE', microbe, 'PRODUCES', 'CHEMICAL', chem_common_name, False))
                            continue

                        if not daily_requests:
                            break

                        try:
                            reactions = good_reactions_xml.findall('Reaction')
                        except:
                            print("no reactions!!")
                            continue

                        chem_in_left = False
                        chem_in_right = False

                        for reaction in reactions:
                            enzrxns = reaction.find('enzymatic-reaction')
                            #if it is not an enzymatic reaction, just continue, fermentation involves enzymes
                            if enzrxns == None:
                                continue
                            enzyme_names = []

                            for enzrxn in enzrxns.findall('Enzymatic-Reaction'):
                                try:
                                    enzyme_names.append(enzrxn.find('common-name').text)
                                except:
                                    continue

                            direction = reaction.findtext('reaction-direction')

                            if not direction:
                                direction = 'LEFT-TO-RIGHT'

                            def get_left_right_ids(direction):
                                lr_ids = []
                                for lr in reaction.findall(direction):
                                    try:
                                        lr_ids.append(lr.find('Compound').get('frameid'))
                                    except:
                                        continue
                                return lr_ids

                            if 'LEFT-TO-RIGHT' in direction or direction == 'REVERSIBLE':
                                left_ids = get_left_right_ids('left')
                                right_ids = get_left_right_ids('right')
                            elif 'RIGHT-TO-LEFT' in direction:
                                left_ids = get_left_right_ids('right')
                                right_ids = get_left_right_ids('left')
                            else:
                                print(f"direction: {direction}")
                                continue

                            new_relation_found = False

                            #for each enzymatic reaction, if the chemical id is found in the left side, it is consumed
                            chem_left_found = any([id in left_ids for id in chem_ids])
                            if chem_left_found:
                                add_relation(('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', chem_common_name, True))
                                chem_in_left = True
                                for right_id in right_ids:
                                    #if the chem appears in both left and right, skip this one
                                    if any([id == right_id for id in chem_ids]):
                                        continue

                                    if biocyc_common_name_mapping.get(right_id):
                                        r_ids, right_common_name = biocyc_common_name_mapping[right_id]
                                    else:
                                        r_ids, right_common_name = get_common_name(right_id)

                                    if not r_ids:
                                        continue

                                    if not chem_common_name == right_common_name:
                                        print(chem_common_name, right_common_name)
                                        add_relation(('CHEMICAL', chem_common_name, 'BECOMES', 'CHEMICAL', right_common_name, True))
                                    new_relation_found = True

                                    add_relation(('MICROBE', microbe, 'PRODUCES', 'CHEMICAL', right_common_name, True))

                                    for enzyme in enzyme_names:
                                        #now we create relations based on the left name and enzyme
                                        add_relation(('MICROBE', microbe, 'SYNTHASIZES', 'ENZYME', enzyme, True))
                                        add_relation(('ENZYME', enzyme, 'CATALYZES', 'CHEMICAL', chem_common_name, True))
                                        new_relation_found = True

                                    if not daily_requests:
                                        break
                                        
                            #if it doesnt appear in the left assume its only a metabolite which gets produced
                            else:
                                pass

                            #now consider when the chemical is on the right side of the reaction, therefore it is produced
                            chem_right_found = any([id in right_ids for id in chem_ids])
                            if chem_right_found:
                                add_relation(('MICROBE', microbe, 'PRODUCES', 'CHEMICAL', chem_common_name, True))
                                chem_in_right = True
                                for left_id in left_ids:
                                    #if the chem appears in both left and right, skip this one
                                    if any([id == left_id for id in chem_ids]):
                                        continue

                                    if biocyc_common_name_mapping.get(left_id):
                                        l_ids, left_common_name = biocyc_common_name_mapping[left_id]
                                    else:
                                        l_ids, left_common_name = get_common_name(left_id)
                                        
                                    if not l_ids:
                                        continue

                                    if not left_common_name == chem_common_name:
                                        print(left_common_name, chem_common_name)
                                        add_relation(('CHEMICAL', left_common_name, 'BECOMES', 'CHEMICAL', chem_common_name, True))
                                        new_relation_found = True

                                    #just because it appears on the left side doesnt necessarily mean it consumes it?
                                    #saved_relations.append(('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', left_common_name, True))

                                    for enzyme in enzyme_names:
                                        #now we create relations based on the left name and enzyme
                                        add_relation(('MICROBE', microbe, 'SYNTHASIZES', 'ENZYME', enzyme, True))
                                        add_relation(('ENZYME', enzyme, 'CATALYZES', 'CHEMICAL', left_common_name, True))
                                        new_relation_found = True

                                    if not daily_requests:
                                        break

                            #just because a chemical isnt produced in this particular reaction doesnt mean it isnt produced in any! dont be selfish bruh
                            else:
                                #do nothing ig ¯\_(ツ)_/¯
                                pass

                            if new_relation_found:
                                print("new relations!")
                                new_relation_found = False

                            if not daily_requests:
                                break

                        #only add false relations if they dont appear in any of them, not just if they didnt appear in a particular one
                        if not chem_in_left:
                            saved_relations.append(('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', chem_common_name, False))
                        if not chem_in_right:
                            saved_relations.append(('MICROBE', microbe, 'PRODUCES', 'CHEMICAL', chem_common_name, False))

                        #utilisation = session.get(f'https://websvc.biocyc.org/getxml?{database}:{chem_id}&fmt=json')               

                with open("list_files\\entity_label_map.json", 'w') as f:
                    json.dump(entity_label_map, f)

    del paper_processed

with open('list_files\\paper_number.json', 'w') as f:
    json.dump(paper_no+2, f)

new_relations_no_dupes = []

for rel in new_relations:
    if rel not in new_relations_no_dupes:
        new_relations_no_dupes.append(rel)

add_cypher_relations_neo(new_relations_no_dupes)

0
no real chemicals found
1
first 5 microbes: [('MICROBE_NAME', 'Enterococcus faecalis', 366, 387), ('MICROBE_NAME', 'Pediococcus pentosaceus', 1701, 1724), ('MICROBE_NAME', 'Streptococcus mutans', 8897, 8917), ('MICROBE_NAME', 'Escherichia coli', 11084, 11100), ('MICROBE_NAME', 'Saccharomyces cerevisiae', 11293, 11317)]
finished processing paper
['Enterococcus faecalis', 'Pediococcus pentosaceus'] ['procyanidins']
microbe: Enterococcus faecalis
microbe: Pediococcus pentosaceus
['Pediococcus pentosaceus'] ['r-lactic acid', 'procyanidins']
microbe: Pediococcus pentosaceus
testing database
new relations!
new relations!
['Pediococcus pentosaceus'] ['r-lactic acid']
microbe: Pediococcus pentosaceus
relation already found. continue
['Streptococcus mutans'] ['lactose', 'r-lactic acid', 'xylitol']
microbe: Streptococcus mutans
testing database
new relations!
testing database
['Streptococcus mutans'] ['ciprofloxacin', 'xylitol', 'r-lactic acid']
microbe: Streptococcus mutans
testing database
[

In [8]:
%%script False

with open('list_files\\saved_relations.json', 'r', encoding='utf-8') as f:
    saved_relations = json.load(f)

add_cypher_relations_neo(saved_relations)

Couldn't find program: 'False'


# <h5>Remove duplicate relations</h5>

In [9]:
%%script False

import json

with open('list_files\\saved_relations.json', 'r', encoding='utf-8') as f:
    saved_relations = json.load(f)

no_repeats = []

for rel in saved_relations:
    if rel not in no_repeats:
        no_repeats.append(rel)

with open('list_files\\saved_relations.json', 'w', encoding='utf-8') as f:
    json.dump(list(no_repeats), f)

Couldn't find program: 'False'


reset

In [10]:
%%script False

import json

with open("list_files\\entity_label_map.json", 'w') as f:
    json.dump({}, f)

with open('list_files\\paper_number.json', 'w') as f:
    json.dump(0, f)

with open('list_files\\saved_relations.json', 'w', encoding='utf-8') as f:
    json.dump([], f)

Couldn't find program: 'False'


In [11]:
%%script False

""" sci_lents = linker(sci_ents)
        linker_labels = []
        for ent in sci_ents.ents:
            #print([linker.kb.cui_to_entity[lent[0]] for lent in ent._.kb_ents])
            if 'CHEMICAL' in ent.label_ and ent.text.lower() not in not_chemicals + [ent.text.lower() for l, text, s, e in mic_ents]:
                if ent._.kb_ents:
                    #print(ent._.kb_ents[0])
                    label = linker.kb.cui_to_entity[ent._.kb_ents[0][0]]
                    linker_labels.append(label.canonical_name)
                else:
                    linker_labels.append(ent.text) 
                    
        linked_sci_ents = [lent for lent in sci_lents.ents if 'CHEMICAL' in lent.label_] """

Couldn't find program: 'False'
